# 03 · Gerçek NAS: one-shot süper ağ eğitimi (`nas_super1`)

**Neden (Karar 6, Furkan 2026-10-16):** Danışman görüşmesinde ayrık makro-mimari ızgara araması (nas_r1b, nas_r2; 12 mimari × 3 çözünürlük) tez için yeterli bulunmadı. Gerçek NAS'a geçilir: **ağırlık paylaşımlı süper ağ** (one-shot) eğitilir, sonra cihaz gecikme tablosuyla (LUT) **evrimsel arama** (NSGA-II) yapılır, seçilen alt ağlar tam COCO'da ince ayarlanır. Bu notebook yalnız süper ağ eğitimidir; arama 03b, ince ayar 03c.

**Arama uzayı** (`exps/nas/supernet.py`; genişlik yok, o budamanın işi): her depthwise konvolüsyonda ayrı ayrı kernel {3, 5, 7} (30 katman) · dark3/dark4 CSP blok sayısı {1, 2, 3} · çözünürlük {320 … 512, adım 32} → ≈ 1,3 × 10¹⁶ alt ağ (eski ızgara: 36).

**Ağırlık paylaşımı:** her DW katmanı tek 7×7 ağırlık taşır, k×k alt kernel merkez kesittir; başlangıçta önceden eğitilmiş 3×3 merkeze konur, kenarlar sıfır → başlangıçta her alt ağ önceden eğitilmiş Nano ile aynı çıktıyı verir (hücre 7'de doğrulanır). Derinlik: CSP'nin ilk d bloğu koşar.

**Tarif:** `train2017 − 5000` (arama için ayrılan 5000 görsel eğitimde yok; val2017 aramada hiç kullanılmaz), batch 64, fp16, çok ölçekli (320–640), lr 0,001/64, ısınma 1 epoch, son 1 epoch mosaic'siz, EMA, önceden eğitilmiş `yolox_nano.pth`'den. **Sandwich kuralı:** her iterasyonda en büyük (k7, d33) + en küçük (k3, d11) + 1 rastgele alt ağ aynı batch'te; kayıplar ortalanır, tek adım. Epoch sayısı (öneri 20) hücre 7'deki pilot süresiyle Furkan'la sabitlenir.

**Erken kontrol:** eğitim 4. epoch sonunda kendini duraklatır; hücre 10 dokunulmamış Nano alt ağını (BN kalibreli, fit1k) başlangıç değeriyle karşılaştırır. Düşüş `EARLY_MAX_DROP` puanı aşarsa durulur ve tarif konuşulur.

**Uzun koşu kuralı:** hücre 1–8 doğrulama/pilot/manifest; **hücre 9 Furkan'ın başlatma onayıyla** koşulur.

**Kesintide:** runtime'ı yeniden bağla → hücre 1–8 (hücre 4'te Drive elle; hücre 7 pilotu tekrarlamaz) → 9 / 10. Drive'daki son checkpoint'ten `--resume`; sandwich alt ağ dizisi iterasyon numarasından türetildiği için aynı kalır.

**Kayıtlar** (`Drive edge_ai/results/nas_super1/`, depoda `results/nas_super1/`, `.pth` hariç): `manifest.json` · `holdout_manifest.json` · `pilot.json` · `runs/supernet/` (`status.json`, `config.json`, `train_log.txt`, `epoch<N>_ckpt.pth`…) · `snapshots.csv` (epoch × alt ağ × fit1k mAP) · `nas_super1_summary.json`.

Kod: `exps/nas/supernet.py`, `scripts/supernet_search.py`, `scripts/make_holdout.py`, `scripts/check_supernet.py`; `scripts/nas_search.py` (manifest, durum, log yardımcıları). YOLOX fork'una dokunulmaz.

In [ ]:
# 1 · Ortam ve sürüm kaydı
import json, platform, subprocess, sys, datetime, os

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
    except Exception as e:
        return f"ERR {e}"

env = {
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "gpu": sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader") or "none",
    "cuda_nvcc": sh("nvcc --version | tail -1"),
    "cpu_count": os.cpu_count(),
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1),
    "disk_free_content": sh("df -h /content | tail -1"),
}
try:
    import torch, torchvision
    env.update(torch=torch.__version__, torchvision=torchvision.__version__,
               torch_cuda=torch.version.cuda, cuda_available=torch.cuda.is_available())
except ImportError:
    env["torch"] = "not installed"
print(json.dumps(env, indent=2))

In [ ]:
# 2 · Kod: depo + YOLOX fork sabit commit (exp ve betik depoda: exps/nas, scripts)
REPO_URL = "https://github.com/Bey3fendi/edge-ai-object-detection.git"
YOLOX_COMMIT = "3ec6263"
WS = "/content/edge_ai_workspace"

import os, subprocess
if not os.path.isdir(WS):
    subprocess.run(["git", "clone", "--recurse-submodules", REPO_URL, WS], check=True)
else:  # runtime aynı kaldıysa depodaki son exp/betik sürümünü al
    subprocess.run(["git", "-C", WS, "pull", "-q", "--ff-only"], check=True)
subprocess.run(["git", "-C", f"{WS}/YOLOX", "fetch", "--all", "-q"], check=True)
subprocess.run(["git", "-C", f"{WS}/YOLOX", "checkout", "-q", YOLOX_COMMIT], check=True)

env["repo_commit"] = sh(f"git -C {WS} rev-parse --short HEAD")
env["yolox_commit"] = sh(f"git -C {WS}/YOLOX rev-parse --short HEAD")
env["yolox_dirty"] = bool(sh(f"git -C {WS}/YOLOX status --porcelain"))
print({k: env[k] for k in ("repo_commit", "yolox_commit", "yolox_dirty")})
assert env["yolox_commit"].startswith(YOLOX_COMMIT)
env["repo_dirty"] = bool(sh(f"git -C {WS} status --porcelain"))
assert not env["repo_dirty"], "depo kirli: kayıtlı commit koşulan kodu temsil etmez"

In [ ]:
# 3 · YOLOX bağımlılıkları
# Not: `pip install -e .` (setup.py develop) güncel setuptools + py3.13'te kırılıyor; YOLOX saf Python
# olduğu için kurulum yerine PYTHONPATH kullanılır. onnx-simplifier eğitimde gerekmez → atlandı.
!pip install -q loguru tabulate thop ninja pycocotools psutil tensorboard opencv-python-headless

import sys, os
YOLOX_DIR = f"{WS}/YOLOX"
if YOLOX_DIR not in sys.path:
    sys.path.insert(0, YOLOX_DIR)
os.environ["PYTHONPATH"] = YOLOX_DIR + ":" + os.environ.get("PYTHONPATH", "")

import importlib.metadata as md
for p in ["torch", "torchvision", "numpy", "opencv-python-headless", "pycocotools", "loguru", "thop"]:
    try:
        env[f"pkg_{p}"] = md.version(p)
    except md.PackageNotFoundError:
        env[f"pkg_{p}"] = None
import yolox
from yolox.exp import get_exp
exp = get_exp(None, "yolox-nano")
print("yolox import OK:", yolox.__file__)
print({k: v for k, v in env.items() if k.startswith("pkg_")})
SCRIPTS = f"{WS}/scripts"
if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)
NAS_EXPS = f"{WS}/exps/nas"
if NAS_EXPS not in sys.path:
    sys.path.insert(0, NAS_EXPS)
import importlib, nas_search as ns, supernet as sn, supernet_search as ss
for _m in (ns, sn, ss):
    importlib.reload(_m)
EXP_FILE = f"{WS}/exps/nas/supernet.py"

In [ ]:
# 4 · Drive + ağırlık (SHA256)
# İlk mount tarayıcıda kullanıcı onayı ister; bağlıysa tekrar denemez.
env.pop("pkg_yolox", None)
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/edge_ai"
DRIVE_DATA = f"{DRIVE_ROOT}/data"
DRIVE_WEIGHTS = f"{DRIVE_ROOT}/weights"
ROUND = "nas_super1"   # gerçek NAS, süper ağ eğitimi (Karar 6, Furkan 2026-10-16)
DRIVE_RESULTS = f"{DRIVE_ROOT}/results/{ROUND}"
for d in (DRIVE_DATA, DRIVE_WEIGHTS, DRIVE_RESULTS):
    os.makedirs(d, exist_ok=True)

import hashlib, urllib.request, shutil
WEIGHT_URL = "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth"
w_drive = f"{DRIVE_WEIGHTS}/yolox_nano.pth"
if not os.path.exists(w_drive):
    urllib.request.urlretrieve(WEIGHT_URL, w_drive)
WEIGHTS = "/content/yolox_nano.pth"
shutil.copy(w_drive, WEIGHTS)

h = hashlib.sha256()
with open(WEIGHTS, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
env["weights_url"] = WEIGHT_URL
env["weights_sha256"] = h.hexdigest()
env["weights_bytes"] = os.path.getsize(WEIGHTS)
print(env["weights_sha256"], env["weights_bytes"])

In [ ]:
# 5 · Veri: tam COCO train2017 + val2017 (Colab yereline)
# train2017 görüntüleri Drive'da yok: Drive'da `data/train2017.zip` varsa oradan, yoksa resmi COCO sunucusundan
# Colab yereline indirilir (≈18 GB). Drive kotası için zip varsayılan olarak Drive'a KAYDEDİLMEZ; her yeni
# runtime'da yeniden indirilir. Anotasyonlar ve val2017 P0'dan beri Drive'da.
# Pilot (hücre 7) için 10k alt kümenin yalnız anotasyon json'u açılır (görselleri train2017'de zaten var).
import zipfile, shutil, time
COCO_LOCAL = "/content/datasets/COCO"
ANN_ZIP = f"{DRIVE_DATA}/annotations_trainval2017.zip"
VAL_ZIP = f"{DRIVE_DATA}/val2017.zip"
SUBSET_ZIP = f"{DRIVE_DATA}/coco_train2017_10k_stratified.zip"
TRAIN_ZIP_DRIVE = f"{DRIVE_DATA}/train2017.zip"
TRAIN_ZIP_URL = "http://images.cocodataset.org/zips/train2017.zip"
SAVE_TRAIN_ZIP_TO_DRIVE = False   # True: ilk indirmede Drive'a da kopyalar (≈18 GB kota)
N_TRAIN = 118_287
for z in (ANN_ZIP, VAL_ZIP, SUBSET_ZIP):
    assert os.path.exists(z), f"{z} yok: önce 00 pilot notebook'unun hücre 5 ve 9'u"
t0 = time.time()
os.makedirs(f"{COCO_LOCAL}/annotations", exist_ok=True)
with zipfile.ZipFile(ANN_ZIP) as z:
    for name in ("annotations/instances_train2017.json", "annotations/instances_val2017.json"):
        if not os.path.exists(f"{COCO_LOCAL}/{name}"):
            z.extract(name, COCO_LOCAL)
with zipfile.ZipFile(SUBSET_ZIP) as z:
    if not os.path.exists(f"{COCO_LOCAL}/annotations/instances_train2017_10k.json"):
        z.extract("annotations/instances_train2017_10k.json", COCO_LOCAL)
if not os.path.isdir(f"{COCO_LOCAL}/val2017"):
    shutil.copy(VAL_ZIP, "/content/val2017.zip")
    with zipfile.ZipFile("/content/val2017.zip") as z:
        z.extractall(COCO_LOCAL)

n_local = len(os.listdir(f"{COCO_LOCAL}/train2017")) if os.path.isdir(f"{COCO_LOCAL}/train2017") else 0
if n_local != N_TRAIN:
    local_zip = "/content/train2017.zip"
    if os.path.exists(TRAIN_ZIP_DRIVE):
        print("train2017.zip Drive'dan kopyalanıyor…", flush=True)
        shutil.copy(TRAIN_ZIP_DRIVE, local_zip)
        env["train_zip_source"] = "drive"
    else:
        print("train2017.zip resmi COCO sunucusundan indiriliyor (≈18 GB)…", flush=True)
        subprocess.run(["wget", "-q", "--show-progress", "--progress=dot:giga", "-O", local_zip + ".part",
                        TRAIN_ZIP_URL], check=True)
        os.replace(local_zip + ".part", local_zip)
        env["train_zip_source"] = TRAIN_ZIP_URL
    env["train_zip_bytes"] = os.path.getsize(local_zip)
    env["train_zip_sha256"] = ns.sha256(local_zip)
    if SAVE_TRAIN_ZIP_TO_DRIVE and not os.path.exists(TRAIN_ZIP_DRIVE):
        shutil.copy(local_zip, TRAIN_ZIP_DRIVE + ".tmp")
        os.replace(TRAIN_ZIP_DRIVE + ".tmp", TRAIN_ZIP_DRIVE)
    print("açılıyor…", flush=True)
    subprocess.run(["unzip", "-q", "-o", local_zip, "-d", COCO_LOCAL], check=True)
    os.remove(local_zip)
env["train_images"] = len(os.listdir(f"{COCO_LOCAL}/train2017"))
env["val_images"] = len(os.listdir(f"{COCO_LOCAL}/val2017"))
env["train_ann_sha256"] = ns.sha256(f"{COCO_LOCAL}/annotations/instances_train2017.json")
env["data_prep_s"] = round(time.time() - t0, 1)
env["disk_free_after_data"] = sh("df -h /content | tail -1")
print({k: env.get(k) for k in ("train_images", "val_images", "train_zip_source", "train_zip_sha256",
                               "train_ann_sha256", "data_prep_s", "disk_free_after_data")})
assert env["train_images"] == N_TRAIN and env["val_images"] == 5000

In [ ]:
# 6 · Arama ayrımı: train2017'den 5000 görsel (fit1k: arama uygunluğu, bn2k: BN kalibrasyonu)
# Tohum sabit (2026) → her runtime'da aynı ayrım; sha256'lar Drive'daki ilk kayıtla karşılaştırılır.
HOLD_SEED = 2026
ann_dir = f"{COCO_LOCAL}/annotations"
hm_local = f"{ann_dir}/nas_holdout_manifest.json"
if not os.path.exists(hm_local):
    stream_out = subprocess.run(["python", f"{WS}/scripts/make_holdout.py", "--coco", COCO_LOCAL,
                                 "--seed", str(HOLD_SEED)], capture_output=True, text=True)
    print(stream_out.stdout[-1500:], stream_out.stderr[-1500:])
    assert stream_out.returncode == 0
hm = json.load(open(hm_local))
hm_drive = ns.read_json(f"{DRIVE_RESULTS}/holdout_manifest.json")
if hm_drive is None:
    ns.write_json(f"{DRIVE_RESULTS}/holdout_manifest.json", hm)
else:
    assert {k: v["sha256"] for k, v in hm["files"].items()} == \
           {k: v["sha256"] for k, v in hm_drive["files"].items()}, "ayrım ilk kayıttan farklı"
assert hm["source_sha256"] == env["train_ann_sha256"] and hm["seed"] == HOLD_SEED
env["holdout_files"] = {k: v["sha256"][:12] for k, v in hm["files"].items()}
print(json.dumps({k: (v["images"], v["annotations"]) for k, v in hm["files"].items()}, indent=1))

In [ ]:
# 7 · Doğrulama + pilot (≈20–25 dk, ≈0,6 CU). Sonuçlar _pilot/ ve pilot.json'a; turun kayıtlarına girmez.
# (a) check_supernet.py önceden eğitilmiş ağırlıkla: başlangıç eşdeğerliği, çıkarma, sandwich, NSGA-II, ONNX.
# (b) Başlangıç değerleri: süper ağ = önceden eğitilmiş Nano; nano/en büyük/en küçük alt ağın fit1k mAP'i
#     (BN kalibreli) → erken kontrolün referansı.
# (c) Pilot: 10k alt kümede 2 epoch süper ağ (her epoch anlık görüntü) → kesinti benzetimi → 3. epoch --resume
#     → epoch1 anlık görüntüsünden alt ağ değerlendirmesi. Kararlı iterasyon süresinden tam bütçe tahmini.
import torch, re, time
r = subprocess.run(["python", f"{WS}/scripts/check_supernet.py", "--weights", WEIGHTS],
                   capture_output=True, text=True, cwd=WS)
print(r.stdout[-2500:]); assert "TÜM DENETİMLER GEÇTİ" in r.stdout, r.stderr[-3000:]

EV = ss.Evaluator(COCO_LOCAL, batch_size=64, workers=os.cpu_count())
init_exp = sn.Exp(); init_exp.nas_pretrained = WEIGHTS
_sup_cpu = init_exp.get_model()
NAMES = init_exp.dw_names
N = len(NAMES)
REF_CFGS = {"nano_416": sn.nano_config(N, 416), "max": sn.max_config(N), "min": sn.min_config(N),
            "nano_320": sn.nano_config(N, 320)}
BATCH, FP16, CU_PER_HOUR = 64, True, 1.54
WORKERS = os.cpu_count()
OUT_ROOT = f"{WS}/YOLOX_outputs/{ROUND}"
def stream(cmd, show, tag):
    return ns.stream(cmd, WS, show, tag)
SHOW_TRAIN = re.compile(ns.SHOW_TRAIN.pattern + r"|Süper ağ")

RUN_PILOT = not os.path.exists(f"{DRIVE_RESULTS}/pilot.json")   # kesintiden sonra pilot tekrarlanmaz
if RUN_PILOT:
    SUP0 = _sup_cpu.cuda().eval()
    init_vals = {}
    for tag, c in REF_CFGS.items():
        init_vals[tag] = ss.evaluate_config(SUP0, NAMES, c, EV)
    print("başlangıç (önceden eğitilmiş Nano ağırlıkları):", {k: v["map5095"] for k, v in init_vals.items()})
    del SUP0; torch.cuda.empty_cache()

    pilot = f"{DRIVE_RESULTS}/_pilot"
    shutil.rmtree(f"{OUT_ROOT}/_pilot", ignore_errors=True); shutil.rmtree(pilot, ignore_errors=True)
    base = ["python", "-u", "YOLOX/tools/train.py", "-f", EXP_FILE, "-d", "1", "-b", str(BATCH), "--fp16",
            "-expn", "_pilot"]
    popts = ["nas_mirror_dir", pilot, "output_dir", OUT_ROOT, "data_num_workers", str(WORKERS),
             "no_aug_epochs", "0", "nas_snapshot_interval", "1", "train_ann", "instances_train2017_10k.json"]
    t_p = time.time()
    txt1, s1 = stream(base + popts + ["nas_pretrained", WEIGHTS, "max_epoch", "2"], SHOW_TRAIN, "pilot 1-2")
    shutil.rmtree(f"{OUT_ROOT}/_pilot"); os.makedirs(f"{OUT_ROOT}/_pilot")
    for n in ("latest_ckpt.pth", "train_log.txt"):
        shutil.copy(f"{pilot}/{n}", f"{OUT_ROOT}/_pilot/{n}")
    txt2, s2 = stream(base + ["--resume"] + popts + ["max_epoch", "3"], SHOW_TRAIN, "pilot resume")
    log = open(f"{pilot}/train_log.txt").read()
    assert "resume training" in log and "start train epoch3" in log
    assert log.count("Süper ağ: {") == 2, "her denemede bir kez beklenir"
    for ep in (1, 2, 3):
        assert torch.load(f"{pilot}/epoch{ep}_ckpt.pth", map_location="cpu")["start_epoch"] == ep
    assert "loss_min" in log and "loss_rnd1" in log, "sandwich kayıpları logda yok"

    # kararlı iterasyon süresi: 3. epoch (resume; tüm giriş boyutları/kernel'ler cudnn önbelleğinde değil, ama
    # epoch 2'nin ikinci yarısı kararlıya en yakın) — YOLOX log satırındaki iter_time ortalamaları
    it = [float(x) for x in re.findall(r"iter_time: ([\d.]+)", log)]
    half = it[len(it) // 2:]
    iter_s = sorted(half)[len(half) // 2]          # medyan (ilk-boyut sıçramalarına dayanıklı)
    iters_full = (118_287 - 5000) // BATCH
    pilot_rec = {"iter_s_median_2nd_half": iter_s, "iters_per_epoch_full": iters_full,
                 "epoch_s_est": round(iter_s * iters_full), "pilot_wall_s": round(time.time() - t_p),
                 "nas_r2_epoch_s_single_net": 886, "measured_utc": ns.now_utc(), "gpu": env["gpu"]}
    for E in (12, 16, 20, 24, 30):
        h = iter_s * iters_full * E / 3600
        pilot_rec[f"est_{E}ep"] = {"hours": round(h, 1), "cu": round(h * CU_PER_HOUR, 1)}

    sup_p = ss.load_supernet(f"{pilot}/epoch1_ckpt.pth")[1]
    pilot_rec["epoch1_fit"] = {t: ss.evaluate_config(sup_p, NAMES, c, EV)["map5095"]
                               for t, c in REF_CFGS.items() if t in ("nano_416", "min")}
    del sup_p; torch.cuda.empty_cache()
    pilot_rec["init_fit"] = {k: v["map5095"] for k, v in init_vals.items()}
    ns.write_json(f"{DRIVE_RESULTS}/pilot.json", pilot_rec)
    print(json.dumps(pilot_rec, indent=1))
else:
    print("pilot.json var, pilot atlandı:", ns.read_json(f"{DRIVE_RESULTS}/pilot.json")["est_20ep"])
del _sup_cpu


In [ ]:
# 8 · Manifest: turun kimlik kartı. MAX_EPOCH pilot sonrası Furkan'la sabitlenir (öneri 20).
from yolox.exp import get_exp
MAX_EPOCH = 20            # ← pilot.json'daki est_*ep satırlarına göre Furkan onayıyla
PAUSE_EPOCH = 4           # erken kontrol duraklaması
EARLY_MAX_DROP = 3.0      # nano_416 fit1k mAP'i başlangıçtan en fazla bu kadar düşebilir (öneri; Furkan onayı)
SNAP = 4
for _m in ("supernet", "nano_macro"):
    sys.modules.pop(_m, None)
_e = get_exp(EXP_FILE, None)
_e.merge(["max_epoch", str(MAX_EPOCH), "nas_snapshot_interval", str(SNAP)])
RECIPE_KEYS = ("max_epoch", "no_aug_epochs", "warmup_epochs", "warmup_lr", "basic_lr_per_img",
               "min_lr_ratio", "scheduler", "weight_decay", "momentum", "ema", "input_size",
               "random_size", "mosaic_prob", "mosaic_scale", "enable_mixup", "mixup_prob",
               "hsv_prob", "flip_prob", "degrees", "translate", "shear", "depth", "width",
               "num_classes", "train_ann", "val_ann", "nmsthre", "seed", "nas_snapshot_interval",
               "nas_sandwich_random", "nas_seed")
NO_AUG = _e.no_aug_epochs
init_fit = ns.read_json(f"{DRIVE_RESULTS}/pilot.json")["init_fit"]
FINGERPRINT = ("round", "runs", "recipe", "space", "eval_protocol", "early_check", "exp_file_sha256",
               "weights_sha256", "train_ann_sha256", "holdout", "yolox_commit")
manifest = ns.ensure_manifest(f"{DRIVE_RESULTS}/manifest.json", {
    "round": ROUND,
    "notebook": "notebooks/03_gercek_nas_supernet_egitimi.ipynb",
    "decisions": "Karar 6 (Furkan, 2026-10-16): gerçek NAS = one-shot süper ağ + LUT + NSGA-II; genişlik "
                 "aramada yok; Colab L4.",
    "runs": ["supernet"],
    "recipe": {**{k: getattr(_e, k, None) for k in RECIPE_KEYS}, "batch": BATCH, "fp16": FP16,
               "devices": 1, "init": "pretrained yolox_nano.pth (3×3 → 7×7 merkez, kenar sıfır)",
               "sandwich": "max + min + nas_sandwich_random rastgele; kayıp ortalaması, tek adım",
               "in_training_eval": False, "train_images": 118_287 - 5000},
    "space": {"kernels": sn.KERNELS, "depth_stages": sn.DEPTH_STAGES, "depths": sn.DEPTHS,
              "resolutions": sn.RESOLUTIONS, "dw_layers": NAMES},
    "eval_protocol": {"split": "train2017 nas_fit1k (1000)", "bn_recalibration": "nas_bn2k (2000), kümülatif",
                      "precision": "fp16", "conf": 0.001, "nms": 0.65, "checkpoint": "EMA",
                      "snapshots_every": SNAP, "ref_configs": {k: sn.config_id(v) for k, v in REF_CFGS.items()}},
    "early_check": {"pause_epoch": PAUSE_EPOCH, "metric": "nano_416 fit1k mAP",
                    "init_value": init_fit["nano_416"], "max_drop": EARLY_MAX_DROP},
    "exp_file_sha256": ns.sha256(EXP_FILE),
    "weights_sha256": env["weights_sha256"], "weights_url": env["weights_url"],
    "train_ann_sha256": env["train_ann_sha256"],
    "holdout": {k: v["sha256"] for k, v in hm["files"].items()},
    "yolox_commit": env["yolox_commit"],
    "repo_commit_at_creation": env["repo_commit"],
    "cu_per_hour": CU_PER_HOUR, "pilot": ns.read_json(f"{DRIVE_RESULTS}/pilot.json"),
    "env_at_creation": env,
}, keys=FINGERPRINT)
del _e
info = {}
from yolox.utils import get_model_info
_s = sn.Exp(); _s.nas_pretrained = WEIGHTS; _sup = _s.get_model().eval()
for tag, c in REF_CFGS.items():
    sub = sn.extract_subnet(_sup, NAMES, c)
    p, g = ns.INFO_RE.search(get_model_info(sub, (c["res"], c["res"]))).groups()
    info[tag] = {"id": sn.config_id(c), "params_m": float(p), "gflops": float(g)}
del _sup, _s
ns.write_json(f"{DRIVE_RESULTS}/runs/supernet/config.json", {"ref_subnets": info, "manifest_round": ROUND})
print(json.dumps({k: manifest[k] for k in ("round", "created_utc", "recipe", "early_check")}, indent=1,
                 default=str))
print(json.dumps(info, indent=1))

In [ ]:
# 9 · Süper ağ eğitimi (UZUN KOŞU — Furkan'ın başlatma onayıyla). Kaldığı yerden devam eder.
# PAUSE_EPOCH'ta kendini durdurur → hücre 10 erken kontrol → bu hücre yeniden çalıştırılınca devam eder.
RUN = "supernet"
def ckpt_epoch(path):
    try:
        return torch.load(path, map_location="cpu")["start_epoch"]
    except Exception as e:
        print(f"  okunamayan checkpoint {path}: {e}")
        return None

def train_supernet(allow_past_pause=False):
    st = ns.get_status(DRIVE_RESULTS, RUN)
    if st["status"] in ("trained", "evaluated"):
        return st
    local, mirror = f"{OUT_ROOT}/{RUN}", f"{DRIVE_RESULTS}/runs/{RUN}"
    shutil.rmtree(local, ignore_errors=True); os.makedirs(local); os.makedirs(mirror, exist_ok=True)
    ep = ckpt_epoch(f"{mirror}/latest_ckpt.pth") if os.path.exists(f"{mirror}/latest_ckpt.pth") else None
    if ep is None:
        for n in os.listdir(mirror):
            if n.endswith("_ckpt.pth") or n == "train_log.txt":
                os.remove(f"{mirror}/{n}")
    if ep is not None and ep >= PAUSE_EPOCH and not allow_past_pause and not st.get("early_check_passed"):
        raise RuntimeError("erken kontrol (hücre 10) henüz geçmedi")
    if ep is None or ep < MAX_EPOCH:
        resume = ep is not None
        if resume:
            for n in ("latest_ckpt.pth", "train_log.txt"):
                shutil.copy(f"{mirror}/{n}", f"{local}/{n}")
        pause = PAUSE_EPOCH if (ep or 0) < PAUSE_EPOCH else 0
        attempt = {"start_utc": ns.now_utc(), "resume_from_epoch": ep, "pause_epoch": pause,
                   "gpu": env["gpu"], "repo_commit": env["repo_commit"]}
        st = ns.set_status(DRIVE_RESULTS, RUN, st, status="training", attempts=st["attempts"] + [attempt])
        cmd = (["python", "-u", "YOLOX/tools/train.py", "-f", EXP_FILE, "-d", "1", "-b", str(BATCH)]
               + (["--fp16"] if FP16 else []) + ["-expn", RUN] + (["--resume"] if resume else [])
               + ["nas_pretrained", "" if resume else WEIGHTS, "nas_mirror_dir", mirror,
                  "output_dir", OUT_ROOT, "data_num_workers", str(WORKERS), "max_epoch", str(MAX_EPOCH),
                  "nas_snapshot_interval", str(SNAP), "nas_pause_epoch", str(pause)])
        _, wall = stream(cmd, SHOW_TRAIN, RUN)
        st["attempts"][-1].update(end_utc=ns.now_utc(), wall_s=round(wall, 1))
        st = ns.set_status(DRIVE_RESULTS, RUN, st)
        ep = ckpt_epoch(f"{mirror}/latest_ckpt.pth")
        if ep < MAX_EPOCH:
            print(f"duraklatıldı: epoch {ep} → hücre 10 (erken kontrol)")
            return st
    log = open(f"{mirror}/train_log.txt").read()
    ns.check_train_log(log, MAX_EPOCH, NO_AUG)
    assert ckpt_epoch(f"{mirror}/last_epoch_ckpt.pth") == MAX_EPOCH
    train_s = round(sum(a.get("wall_s") or 0 for a in st["attempts"]), 1)
    return ns.set_status(DRIVE_RESULTS, RUN, st, status="trained", train_s=train_s,
                         trained_utc=ns.now_utc(), repo_commit=env["repo_commit"])

st = train_supernet()
print(RUN, st["status"], "deneme", len(st["attempts"]), "eğitim", st.get("train_s"))

In [ ]:
# 10 · Anlık görüntü değerlendirmesi (fit1k, BN kalibreli): nano_416 / nano_320 / max / min alt ağları
# Erken kontrol: PAUSE_EPOCH anlık görüntüsünde nano_416 düşüşü ≤ EARLY_MAX_DROP → status'a işlenir, hücre 9 devam.
import csv
def snapshot_eval(epoch):
    st = ns.get_status(DRIVE_RESULTS, RUN)
    key = f"e{epoch}"
    done = st.get("snapshots", {}).get(key, {})
    path = f"{DRIVE_RESULTS}/runs/{RUN}/" + ("last_epoch_ckpt.pth" if epoch == MAX_EPOCH else f"epoch{epoch}_ckpt.pth")
    assert ckpt_epoch(path) == epoch, path
    sup = None
    for tag, c in REF_CFGS.items():
        if tag in done:
            continue
        sup = sup or ss.load_supernet(path)[1]
        done[tag] = ss.evaluate_config(sup, NAMES, c, EV)
        st = ns.set_status(DRIVE_RESULTS, RUN, st, snapshots={**st.get("snapshots", {}), key: done})
    del sup; torch.cuda.empty_cache()
    return st

def rebuild_snapshots():
    st = ns.get_status(DRIVE_RESULTS, RUN)
    rows = [{"epoch": int(k[1:]), "subnet": t, "config_id": sn.config_id(REF_CFGS[t]), **v}
            for k, d in st.get("snapshots", {}).items() for t, v in d.items()]
    rows += [{"epoch": 0, "subnet": t, "config_id": sn.config_id(REF_CFGS[t]), "map5095": v}
             for t, v in manifest["pilot"]["init_fit"].items()]
    rows.sort(key=lambda r: (r["epoch"], r["subnet"]))
    fields = ["epoch", "subnet", "config_id", "map5095", "ap50", "bn_s", "eval_s", "evaluated_utc"]
    tmp = f"{DRIVE_RESULTS}/snapshots.csv.tmp"
    with open(tmp, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore"); w.writeheader(); w.writerows(rows)
    os.replace(tmp, f"{DRIVE_RESULTS}/snapshots.csv")
    return rows

mirror = f"{DRIVE_RESULTS}/runs/{RUN}"
avail = sorted(e for e in range(SNAP, MAX_EPOCH + 1, SNAP)
               if os.path.exists(f"{mirror}/epoch{e}_ckpt.pth") or
               (e == MAX_EPOCH and os.path.exists(f"{mirror}/last_epoch_ckpt.pth")
                and ckpt_epoch(f"{mirror}/last_epoch_ckpt.pth") == MAX_EPOCH))
for e in avail:
    st = snapshot_eval(e)
    print(f"epoch {e}:", {t: v["map5095"] for t, v in st["snapshots"][f"e{e}"].items()})
rebuild_snapshots()
st = ns.get_status(DRIVE_RESULTS, RUN)
if f"e{PAUSE_EPOCH}" in st.get("snapshots", {}) and not st.get("early_check_passed"):
    v = st["snapshots"][f"e{PAUSE_EPOCH}"]["nano_416"]["map5095"]
    ref = manifest["early_check"]["init_value"]
    print(f"ERKEN KONTROL: nano_416 fit1k {v} (başlangıç {ref}, izin verilen düşüş {EARLY_MAX_DROP})")
    assert v >= ref - EARLY_MAX_DROP, "süper ağ önceden eğitilmiş bilgiyi bozuyor: dur, tarif konuşulmalı"
    ns.set_status(DRIVE_RESULTS, RUN, st, early_check_passed=True, early_check={"value": v, "ref": ref})
    print("erken kontrol geçti → hücre 9'u yeniden çalıştır (kalan epoch'lar)")

In [ ]:
# 11 · Özet: anlık görüntü eğrisi + süre/CU → nas_super1_summary.json, nas_super1_snapshots.png
import pandas as pd, matplotlib.pyplot as plt
rows = rebuild_snapshots()
df = pd.DataFrame(rows)
print(df.pivot_table(index="epoch", columns="subnet", values="map5095").to_string())
st = ns.get_status(DRIVE_RESULTS, RUN)
cu = (st.get("train_s") or 0) / 3600 * CU_PER_HOUR
ns.write_json(f"{DRIVE_RESULTS}/{ROUND}_summary.json", {
    "round": ROUND, "written_utc": ns.now_utc(), "status": st["status"], "train_s": st.get("train_s"),
    "cu_train": round(cu, 2), "attempts": len(st["attempts"]), "early_check": st.get("early_check"),
    "snapshots": st.get("snapshots")})
fig, ax = plt.subplots(figsize=(7, 4.5))
for t, g in df.groupby("subnet"):
    g = g.sort_values("epoch"); ax.plot(g["epoch"], g["map5095"], marker="o", label=t)
ax.set_xlabel("epoch (süper ağ)"); ax.set_ylabel("mAP@[.5:.95] fit1k (BN kalibreli)")
ax.set_title("nas_super1: referans alt ağlar"); ax.legend(); fig.tight_layout()
fig.savefig(f"{DRIVE_RESULTS}/{ROUND}_snapshots.png", dpi=150); plt.show()
print(f"eğitim {st.get('train_s')} sn ≈ {cu:.1f} CU")